<!-- ## API AEROFLIGTH -->

<!-- ### endpoint="https://aerodatabox.p.rapidapi.com/airports/{codeType}/{code}/delays"
### Returns: Statistical delay information about delays (median delay, delay index, cancelled flights) of arrivals and departures for the requested airport,
### represented by:
-   a single AirportDelayContract item displaying the delay information based on flight movements within the 2 hours prior to the `current moment`, if no dateLocal is specified;
-   a single AirportDelayContract item displaying the delay information based on flight movements within the 2 hours prior to the `moment requested in dateLocal`#, if dateLocal is specified;  

#### If codeType is:
-   icao, then this field must be a 4-character ICAO-code of the airport (e.g.: EHAM, KLAX, UUEE, etc.);
-   iata, then this field must be a 3-character IATA-code of the airport (e.g.: AMS, SFO, LAX, etc.). -->

<!-- #### departuresDelayInformation  
object 
required  
Delay information about a batch of flights

*  numTotal  
integer<int32> 
required  
Total number of flights in the the batch (including cancelled)

*  numQualifiedTotal  
integer<int32> 
required  
Total number of flights in the batch, which were used to to calculate the delay information (including cancelled). Should equal to or less than NumTotal.Show all...

*  numCancelled  
integer<int32> 
required  
Total amount of flights in the batch

*  medianDelay  
string<date-span> or null  
Median delay of flights in the batch (format: [-]hh:mm:ss). Value can be negative (therefore, means early occurence).

*  delayIndex  
number<float> or null  
Normalized value on scale from 0.0 to 5.0 which corresponds with current amount of delays and cancellations in a given batch of flights (the less - the better).

#### arrivalsDelayInformation  
object  
required  
Delay information about a batch of flights

*  numTotal  
integer<int32>
required  
Total number of flights in the the batch (including cancelled)

*  numQualifiedTotal  
integer<int32>
required  
Total number of flights in the batch, which were used to to calculate the delay information (including cancelled). Should equal to or less than NumTotal.Show all...

*  numCancelled  
integer<int32>
required  
Total amount of flights in the batch

*  medianDelay  
string<date-span> or null  
Median delay of flights in the batch (format: [-]hh:mm:ss). Value can be negative (therefore, means early occurence).

*  delayIndex  
number<float> or null  
Normalized value on scale from 0.0 to 5.0 which corresponds with current amount of delays and cancellations in a given batch of flights (the less - the better). -->

<!-- ***** FIN TEST -->

# Imports


In [ ]:
from datetime import datetime, timedelta
import time
from tqdm import tqdm

import requests

from sqlalchemy import Table, MetaData,create_engine,text,exc
from sqlalchemy.dialects.postgresql import insert as pg_insert

import awswrangler as wr
import boto3

from io import BytesIO,StringIO

import pandas as pd
import os

import logging

from dotenv import load_dotenv
load_dotenv()


# Variables d'environnement

In [ ]:

# =====================Configuration du niveau de LOGGING====================
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')



# ====================== VARIABLES ENVIRONNEMENT AWS S3======================
aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID_EQUIPE")
aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY_EQUIPE")
region_name=os.getenv("AWS_DEFAULT_REGION_EQUIPE")

# ====================== CONFIGURATION API AERODATABOX (vols aeroports) ======================

API_HOST = "https://aerodatabox.p.rapidapi.com"
HEADERS = {
    "X-RapidAPI-Key": os.getenv("X_RAPIDAPI_KEY"),
    "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
}
AEROPORTS = {
    "LFPG": "CDG",
    "LFPO": "ORY",
    "LFMN": "NCE",
    "LFLL": "LYS",
    "LFML": "MRS"
}
# Coordonnées GPS des aéroports pour météo

COORDS_AEROPORTS = {
    "LFPG": (49.0097, 2.5478),   # CDG
    "LFPO": (48.7253, 2.3594),   # ORY
    "LFMN": (43.6653, 7.2150),   # NCE
    "LFLL": (45.7264, 5.0908),   # LYS
    "LFML": (43.4393, 5.2214)    # MRS
}


# Paramètres de l'API pour inclure tous les types de vols et les détails nécessaires

params = {
    "withLeg": "true",
    "direction": "Both",
    "withCancelled": "true",
    "withCodeshared": "true",
    "withCargo": "true",
    "withPrivate": "false"
}

# ====================== VARIABLES ENVIRONNEMENT NEONDB - BASE DE DONNÉES ======================
DB_USER = os.getenv("PPML_NEON_USER_ADMIN")        
DB_PASSWORD = os.getenv("PPML_NEON_PASS_ADMIN")  
DB_HOST = os.getenv("PPML_NEON_HOST_ADMIN")
DB_PORT = os.getenv("PPML_NEON_PORT")
DB_NAME = os.getenv("PPML_NEON_DBNAME_ADMIN")

logging.info("Variables d'environnement chargées :")
logging.info(f"DBHOST: {DB_HOST}")
logging.info(f"DBNAME: {DB_NAME}")


# ====================== creation engine SQLAlchemy et connexion à la base de données ======================
DB_URL = f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(
    DB_URL,
    echo=False,
    # Paramètres essentiels pour Neon (scale-to-zero + pooler)
    pool_pre_ping=True,      # Vérifie si la connexion est encore vivante avant chaque requête
    pool_recycle=300,        # Recycle les connexions toutes les 5 minutes (important !)
    pool_size=8,             # Taille du pool (tu peux monter à 10-15 si besoin)
    max_overflow=10,
    connect_args={
        "sslmode": "require",
        "channel_binding": "require",   # Tu peux garder si tu veux
        "keepalives": 1,
        "keepalives_idle": 30,
        "keepalives_interval": 10,
        "keepalives_count": 5
    }
)
#on va tester la connexion à la base de données en exécutant une requête simple (SELECT 1) pour s'assurer que tout est configuré correctement
try:
    with engine.connect() as connection:
        result = connection.execute(text("SELECT 1"))
        logging.info(f"Test de connexion à la base de données : {result.scalar()}")
except exc.SQLAlchemyError as e:
    logging.error(f"Erreur lors du test de connexion à la base de données : {e}")
else:   
    logging.info("Test de connexion à la base de données réussi !")

# ====================== CONFIGURATION PROJET ======================

nb_jours = 179 #(+1 avec meteo de la veille => -180 pour couvrir 6 mois complets)


end_date = (datetime.now() - timedelta(days=1)).replace(hour=23, minute=59, second=59, microsecond=999999)
start_date = (end_date - timedelta(days= nb_jours)).replace(hour=0, minute=0, second=0, microsecond=0)
start_meteo_date=start_date- timedelta(days= 1) 
start_airports_delays=start_date- timedelta(days= 1) 


logging.info(f"Date de début pour les données de mouvements : {start_date.strftime('%Y-%m-%d %H:%M')}")
logging.info(f"Date de début pour les données météo : {start_meteo_date.strftime('%Y-%m-%d %H:%M')}")
logging.info(f"Date de début pour les données airport delays : {start_airports_delays.strftime('%Y-%m-%d %H:%M')}") 
logging.info(f"Date de fin pour les données de mouvements  : {end_date.strftime('%Y-%m-%d %H:%M')}")
logging.info(f"Nombre de jours à récupérer : {nb_jours}")


# UTILITAIRES

### Pour Dataframes

In [ ]:
def concatene_liste_df(liste_df,unique_key):
    df_concat = pd.concat(liste_df, ignore_index=True)
    df_concat = df_concat.drop_duplicates(
        # Clé unique exemple : combinaison de icao + flight_number + scheduled_utc + type (departure/arrival)
        subset=unique_key,
        keep='last'
    )
    return df_concat

### Pour AWS S3

*   Sauvegarder sur S3  
Dataframe Panda "df_mouvs" => fichier "filename.parquet" sur S3

In [ ]:
def sauvegarder_s3(df, filename):
    # Créer une session boto3
    session = boto3.Session(
        aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID_EQUIPE"),
        aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY_EQUIPE"),
        region_name=os.getenv("AWS_DEFAULT_REGION_EQUIPE", "eu-north-1")
    )
    bucket_name = os.getenv("BUCKET_EQUIPE")
    logging.info(f"Début sauvegarde S3 : {filename} dans le bucket {bucket_name}")
    try:
        wr.s3.to_parquet(
            df=df,
            path=f's3://{bucket_name}/{filename}',  # chemin complet dans le bucket
            compression='snappy',
            index=False,
            boto3_session=session   # ici on passe la session
        )
        logging.info(f"Sauvegarde S3 terminée :  {filename} dans le bucket {bucket_name}")
    except Exception as e:
        logging.error(f"Erreur lors de la sauvegarde S3 : {e}")

In [ ]:
# =============================================
# === UTILITAIRES S3 - VÉRIFICATION & NETTOYAGE ===
# =============================================

import boto3
import awswrangler as wr
import logging
import os

def lister_fichiers_s3(prefix=""):
    """
    Liste tous les fichiers dans le bucket S3 de l'équipe.
    """
    session = boto3.Session(
        aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID_EQUIPE"),
        aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY_EQUIPE"),
        region_name=os.getenv("AWS_DEFAULT_REGION_EQUIPE", "eu-north-1")
    )
    bucket_name = os.getenv("BUCKET_EQUIPE")
    
    if not bucket_name:
        logging.error("BUCKET_EQUIPE non défini dans les variables d'environnement")
        return []
    
    try:
        path = f's3://{bucket_name}/{prefix}'.rstrip('/')
        objects = wr.s3.list_objects(path=path, boto3_session=session)
        
        if not objects:
            print(f"✅ Aucun fichier trouvé dans {path}")
            return []
        
        print(f"📁 {len(objects)} fichier(s) trouvé(s) dans {path}")
        
        for obj in sorted(objects):
            if isinstance(obj, dict):
                key = obj.get('Key') or obj.get('key', 'Unknown')
                size = obj.get('Size') or obj.get('size', 0)
            else:
                key = str(obj)
                size = 0
            
            size_mb = size / (1024 * 1024) if size > 0 else 0
            print(f"   • {key}  ({size_mb:.2f} MB)")
        
        return objects
        
    except Exception as e:
        logging.error(f"Erreur lors du listage S3 : {e}")
        print(f"❌ Erreur listage S3 : {e}")
        return []


def supprimer_fichier_s3(filename, confirmation=True):
    """
    Supprime UN SEUL fichier spécifique dans le bucket S3.
    
    Args:
        filename (str): Nom du fichier ou chemin complet (ex: 'airport_trafic.parquet' ou 'folder/file.parquet')
        confirmation (bool): Demande confirmation avant suppression
    """
    session = boto3.Session(
        aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID_EQUIPE"),
        aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY_EQUIPE"),
        region_name=os.getenv("AWS_DEFAULT_REGION_EQUIPE", "eu-north-1")
    )
    bucket_name = os.getenv("BUCKET_EQUIPE")
    
    if not bucket_name:
        logging.error("BUCKET_EQUIPE non défini")
        return False
    
    # Construction du chemin complet
    path = f's3://{bucket_name}/{filename}'.rstrip('/')
    
    try:
        # Vérifier si le fichier existe
        files = wr.s3.list_objects(path=path, boto3_session=session)
        
        if not files:
            print(f"⚠️ Fichier non trouvé : {path}")
            return False
        
        print(f"🗑️  Fichier trouvé : {path}")
        
        if confirmation:
            reponse = input(f"\n❗ Voulez-vous vraiment supprimer ce fichier ? (oui/non) : ").strip().lower()
            if reponse not in ['oui', 'yes', 'o', 'y']:
                print("❌ Suppression annulée.")
                return False
        
        # Suppression du fichier
        wr.s3.delete_objects(path=path, boto3_session=session)
        logging.info(f"✅ Fichier supprimé : {filename}")
        print(f"✅ Fichier supprimé avec succès : {filename}")
        return True
        
    except Exception as e:
        logging.error(f"Erreur lors de la suppression du fichier {filename} : {e}")
        print(f"❌ Erreur suppression : {e}")
        return False


def nettoyer_s3(prefix="", confirmation=True):
    """
    Supprime TOUS les fichiers sous un préfixe (dossier).
    """
    session = boto3.Session(
        aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID_EQUIPE"),
        aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY_EQUIPE"),
        region_name=os.getenv("AWS_DEFAULT_REGION_EQUIPE", "eu-north-1")
    )
    bucket_name = os.getenv("BUCKET_EQUIPE")
    
    if not bucket_name:
        logging.error("BUCKET_EQUIPE non défini")
        return False
    
    path = f's3://{bucket_name}/{prefix}'.rstrip('/')
    
    try:
        objects = wr.s3.list_objects(path=path, boto3_session=session)
        
        if not objects:
            print(f"✅ Aucun fichier à supprimer dans {path}")
            return True
        
        print(f"⚠️  {len(objects)} fichier(s) vont être supprimés dans : {path}")
        for obj in objects[:10]:
            key = obj.get('Key') if isinstance(obj, dict) else str(obj)
            print(f"   • {key}")
        if len(objects) > 10:
            print(f"   ... et {len(objects)-10} autres")
        
        if confirmation:
            reponse = input("\n❗ Confirmez-vous la suppression de TOUS ces fichiers ? (oui/non) : ").strip().lower()
            if reponse not in ['oui', 'yes', 'o', 'y']:
                print("❌ Suppression annulée.")
                return False
        
        wr.s3.delete_objects(path=path, boto3_session=session)
        logging.info(f"✅ {len(objects)} fichiers supprimés dans {path}")
        print(f"✅ Suppression terminée avec succès ({len(objects)} fichiers).")
        return True
        
    except Exception as e:
        logging.error(f"Erreur lors du nettoyage S3 : {e}")
        print(f"❌ Erreur suppression : {e}")
        return False

In [ ]:
# UTILE PENDANT LES TESTS POUR NETTOYER LE BUCKET S3 DE L'EQUIPE AVANT DE LANCER LES EXTRACTIONS
# nettoyer_s3(prefix="", confirmation=True)

# UTILE POUR VÉRIFIER LE CONTENU DU BUCKET S3 DE L'EQUIPE
# lister_fichiers_s3()

#### API

*   call API

In [ ]:
# ====================== FONCTIONS API======================
#cette fonction est un wrapper classique pour faire les appels à l'API 
# avec gestion des erreurs basique et respect du rate limit
def appeler_api(url, HEADERS,params):
    """Fonction simple pour appeler l'API avec protection d'erreurs"""
    try:
        reponse = requests.get(url, headers=HEADERS, params=params, timeout=20)
        reponse.raise_for_status()
        return reponse.json()
    except Exception as e:
        print(f"    ERREUR API : {e}")
        return None

#### FICHIERS

In [ ]:

# generateur de timestamp pour les nom de fichiers
def get_timestamp():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

## SCRIPTS SQLalchemy

#### Scripts creation des tables 

* table "mouv_aeroport" (mouvements par aeroports)

In [ ]:
def init_table_mouvements_aeroports(engine):
    req = text("""
        CREATE TABLE IF NOT EXISTS mouvements_aeroports (
            id                  BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
            icao                TEXT NOT NULL,
            type                TEXT NOT NULL,
            flight_number       TEXT,
            status              TEXT,
            airline             TEXT,
            scheduled_utc       TIMESTAMPTZ,
            revised_utc         TIMESTAMPTZ,
            runway_utc          TIMESTAMPTZ,
            delay_minutes       DOUBLE PRECISION,
            terminal_dep        TEXT,
            terminal_arr        TEXT,
            destination_icao    TEXT,

            CONSTRAINT unique_vol UNIQUE (icao, flight_number, scheduled_utc, type)
        );

        CREATE INDEX IF NOT EXISTS idx_mouvs_icao_scheduled ON mouvements_aeroports (icao, scheduled_utc);
        CREATE INDEX IF NOT EXISTS idx_mouvs_flight_number   ON mouvements_aeroports (flight_number);
    """)

    with engine.begin() as conn:
        conn.execute(req)

    logging.info("✅ Table mouvements_aeroports créée avec GENERATED BY DEFAULT AS IDENTITY")

*   table "meteo_aeroports" (meteo par aéroport)

In [ ]:
def init_table_meteo_aeroports(engine):

    ## Création de la table avec la contrainte UNIQUE sur tes 4 colonnes
    req_create_table = text("""
        CREATE TABLE IF NOT EXISTS meteo_aeroports (
            id                      SERIAL PRIMARY KEY,
            time                    TIMESTAMPTZ NOT NULL,
            temperature_2m          DOUBLE PRECISION,
            relative_humidity_2m    DOUBLE PRECISION,
            wind_speed_10m          DOUBLE PRECISION,
            pressure_msl            DOUBLE PRECISION,
            precipitation           DOUBLE PRECISION,
            cloud_cover             DOUBLE PRECISION,
            wind_gusts_10m          DOUBLE PRECISION,
            icao                 TEXT NOT NULL,
            
            -- CONTRAINTE UNIQUE pour empêcher les doublons
            CONSTRAINT unique_meteo 
                UNIQUE (time, icao)

        );

        -- Index pour accélérer les recherches par aéroport et date de vol
        CREATE INDEX IF NOT EXISTS idx_meteo_icao_time 
            ON meteo_aeroports (icao, time);
    """
    )
    # Exécution de la requête
    with engine.connect() as conn:
        conn.execute(req_create_table)
        conn.commit()                     # Important pour PostgreSQL

    ## INFOS
    logging.info("Table 'meteo_aeroports' créée si inexistante, avec contrainte UNIQUE.")
    logging.info("   Clé unique : (time, icao)\n")

In [ ]:
def init_table_airport_delays(engine):
    """Création de la table airport_delays_historical avec les bons noms de colonnes"""

    req_create_table = text("""
        CREATE TABLE IF NOT EXISTS airport_delays (
            id                      SERIAL PRIMARY KEY,
            icao                    TEXT NOT NULL,
            
            -- Fenêtre temporelle
            window_from_utc         TIMESTAMPTZ NOT NULL,
            window_to_utc           TIMESTAMPTZ,
            window_from_local       TEXT,
            window_to_local         TEXT,
            
            -- Départs
            dep_num_total           INTEGER,
            dep_num_qualified       INTEGER,
            dep_num_cancelled       INTEGER,
            dep_median_delay        INTERVAL,           -- ex: '00:15:30' ou '-00:05:00'
            dep_delay_index         DOUBLE PRECISION,
            
            -- Arrivées
            arr_num_total           INTEGER,
            arr_num_qualified       INTEGER,
            arr_num_cancelled       INTEGER,
            arr_median_delay        INTERVAL,
            arr_delay_index         DOUBLE PRECISION,

            -- CONTRAINTE UNIQUE pour empêcher les doublons
            CONSTRAINT unique_airport_delays 
                UNIQUE (icao, window_from_utc)
        );

        -- Index pour accélérer les recherches
        CREATE INDEX IF NOT EXISTS idx_airport_delays_icao_from 
            ON airport_delays (icao, window_from_utc);
            
        CREATE INDEX IF NOT EXISTS idx_airport_delays_from_utc 
            ON airport_delays (window_from_utc);
    """)

    # Exécution de la requête
    with engine.connect() as conn:
        try:
            conn.execute(req_create_table)
            conn.commit()
            logging.info("✅ Table 'airport_delays' créée ou déjà existante.")
            logging.info("   Clé unique : (icao, window_from_utc)")
            logging.info("   Colonnes ajoutées : window_from_utc, dep_median_delay_min, arr_median_delay_min\n")
        except Exception as e:
            logging.error(f"Erreur lors de la création de la table airport_delays : {e}")
            conn.rollback()

#### Scripts insertions dans les tables 

*   Script insertions dans la table "mouv_aeroport" (mouvements par aeroports)

In [ ]:
def inserer_base(engine, df, table_name, conflict_columns=None):
    """
    Insert DataFrame into PostgreSQL table with ON CONFLICT DO NOTHING
    
    Parameters:
        conflict_columns: list[str] | None
            Colonnes qui définissent l'unicité (ON CONFLICT).
            Si None → on tente d'utiliser les colonnes PK ou on passe sans conflit.
    """
    logging.info(f"Insertion dans la table {table_name}...")

    if df.empty:
        logging.warning(f"DataFrame vide pour la table {table_name}")
        return

    # Normalisation des colonnes
    df = df.copy()
    df.columns = df.columns.str.lower()

    # Chargement de la structure de la table
    metadata = MetaData()
    table = Table(table_name, metadata, autoload_with=engine)

    data = df.to_dict(orient='records')

    with engine.begin() as conn:
        # Utilisation de l'insert PostgreSQL
        stmt = pg_insert(table).values(data)

        if conflict_columns:
            # Version avec gestion des doublons
            stmt = stmt.on_conflict_do_nothing(
                index_elements=conflict_columns
            )
        else:
            # Si pas de colonnes spécifiées → ON CONFLICT DO NOTHING simple
            # (PostgreSQL utilisera les contraintes uniques/PK existantes)
            stmt = stmt.on_conflict_do_nothing()

        conn.execute(stmt)
        logging.info(f"✅ {len(data):,} enregistrements traités dans {table_name}")

In [ ]:

def inserer_mouvements_aeroports(engine, df_mouvs):
    if df_mouvs.empty:
        return

    # Nettoyage obligatoire
    if 'id' in df_mouvs.columns:
        df_mouvs = df_mouvs.drop(columns=['id'])

    # Conversion dates
    for col in ['scheduled_utc', 'revised_utc', 'runway_utc']:
        if col in df_mouvs.columns:
            df_mouvs[col] = pd.to_datetime(df_mouvs[col], utc=True, errors='coerce')

    try:
        with engine.begin() as conn:
            df_mouvs.to_sql(
                name="mouvements_aeroports",
                con=conn,
                if_exists='append',     # PAS de 'replace' ici
                index=False,
                chunksize=40000,
                method='multi'
            )
        logging.info(f"✅ {len(df_mouvs):,} lignes insérées avec id auto-généré")
    except Exception as e:
        logging.error(f"Erreur insertion : {e}")
        raise

### Table documentaire informative sur les colonnes de toutes les tables:
"documentation_colonnes"

In [ ]:
# ==================== CRÉATION DE LA TABLE DE DOCUMENTATION ====================
create_doc_table = text("""
    CREATE TABLE IF NOT EXISTS documentation_colonnes (
        table_name          TEXT,
        position            INTEGER,
        column_name         TEXT,
        data_type           TEXT,
        max_length          INTEGER,
        is_nullable         TEXT,
        default_value       TEXT,
        description         TEXT,           -- ← Ici tu mettras le court descriptif
        PRIMARY KEY (table_name, column_name)
    );
""")

with engine.begin() as conn:
    conn.execute(create_doc_table)

print("✅ Table 'documentation_colonnes' créée ou déjà existante.")

## SCRIPTS DONNEES BRUTES
### API AEROPORTS (AeroDataBox)

#### Script formattage des données depuis la request API AeroDataBox

In [ ]:
def extraire_vols(donnees_json, icao):
    """Parsing adapté à la structure de l'API"""
    if not donnees_json:
        return pd.DataFrame()
    
    # on va juste créer une colonne calculée pour recupérer directement le retard de chaque avion  
    rows = []
    # l'API AeroDataBox renvoie deux listes séparées dans le JSON : "departures" et "arrivals"
    for mov_type in ["departures", "arrivals"]:
        for flight in donnees_json.get(mov_type, []):
            departure = flight.get("departure", {})
            arrival = flight.get("arrival", {})
            
            scheduledTime = departure.get("scheduledTime", {}).get("utc")
            revisedTime   = departure.get("revisedTime",   {}).get("utc")
            runwayTime    = departure.get("runwayTime",    {}).get("utc")
            
            delay_min = None
            if scheduledTime and revisedTime:
                try:
                    s = datetime.fromisoformat(scheduledTime.replace("Z", "+00:00"))
                    r = datetime.fromisoformat(revisedTime.replace("Z", "+00:00"))
                    delay_min = round((r - s).total_seconds() / 60, 1)
                except:
                    pass
            
            row = {
                #ajout de l'icao pour pouvoir faire le lien avec les autres tables en base de données
                "icao": icao,
                "type": "departure" if mov_type == "departures" else "arrival",
                "flight_number": flight.get("number"),
                "status": flight.get("status"),
                "airline": flight.get("airline", {}).get("name"),
                "scheduled_utc": scheduledTime,
                "revised_utc": revisedTime,
                "runway_utc": runwayTime,
                # Ajout de la colonne calculée simple
                "delay_minutes": delay_min,
                #chaque liste dans l'API a le meme nom de colonne pour depart
                "terminal_dep": departure.get("terminal"),
                "terminal_arr": arrival.get("terminal"),
                "destination_icao": arrival.get("airport", {}).get("icao") if mov_type == "departures" else None,
            }
            rows.append(row)
    
    return pd.DataFrame(rows)

#### Script extraction des données API AeroDataBox

In [ ]:
def get_mouvements_aeroports_historical(AEROPORTS, start_date: datetime, end_date: datetime, HEADERS, params, engine):
    """
    Fonction principale pour récupérer les vols des 5 aéroports entre start_date et end_date.
    Deux appels par jour pour couvrir 24h.
    """
    #print(f"Début de la récupération des données pour 5 aéroports de {start_date.strftime('%Y-%m-%d')} à {end_date.strftime('%Y-%m-%d')}\n")
    print(f"Début de la récupération des données pour 5 aéroports de {start_date} à {end_date}\n")


    unique_key = ['icao', 'flight_number', 'scheduled_utc', 'type']
    
    # Calcul du nombre de jours (inclusif)
    nb_jours = (end_date.date() - start_date.date()).days + 1
    print(f"Nombre de jours à traiter : {nb_jours}\n")
    jour_actuel = start_date

    for i in range(nb_jours):
        date_str = jour_actuel.strftime("%Y-%m-%d")
        print(f"  Jour {date_str}  ({i+1}/{nb_jours})")
        
        # Deux tranches par jour (00:00-11:59 et 12:00-23:59)
        for heure_debut in ["00:00", "12:00"]:
            debut = f"{date_str}T{heure_debut}"
            fin   = f"{date_str}T{'11:59' if heure_debut == '00:00' else '23:59:59'}"
            
            for icao, nom in AEROPORTS.items():   # On boucle sur les aéroports ici pour plus de clarté
                url = f"{API_HOST}/flights/airports/icao/{icao}/{debut}/{fin}"

                try:
                    donnees = appeler_api(url, HEADERS, params)
                    
                    if donnees:
                        df_temp = extraire_vols(donnees, icao)
                        
                        if not df_temp.empty:
                            # Insertion directe tranche par tranche (plus sûr pour les grosses périodes)
                            inserer_base(engine, df_temp, "mouvements_aeroports", conflict_columns=unique_key)
                            print(f"    ✓ {len(df_temp)} vols récupérés pour {nom} ({debut} → {fin})")
                        else:
                            print(f"    ✗ Aucune donnée pour {nom} ({debut} → {fin})")
                    else:
                        print(f"    ✗ Pas de réponse API pour {nom} ({debut} → {fin})")
                
                except Exception as e:
                    logging.error(f"Erreur récupération vols {nom} ({debut} → {fin}) : {e}")
                
                time.sleep(2)  # rate limit

        jour_actuel += timedelta(days=1)

    logging.info("=== Récupération des mouvements pour 5 AÉROPORTS TERMINÉE ===")
    logging.info(f"Période traitée : du {start_date.strftime('%Y-%m-%d')} au {end_date.strftime('%Y-%m-%d')}")
    
    return None

AeroDataBox Get Aeroports DELAYS

In [ ]:
def get_airport_delays_historical(
    coord_aeroport: dict,
    start_date: datetime,
    end_date: datetime,
    engine
):
    """
    insère uniquement les données brutes de l'API Airport Delays.
    Aucune conversion, aucune colonne supplémentaire.
    """
    logging.info(f"Récupération delays bruts du {start_date.strftime('%Y-%m-%d')} "
                 f"au {end_date.strftime('%Y-%m-%d %H:%M')}")

    icaos = list(coord_aeroport.keys())
    total_inserted = 0

    current_date = start_date.date()
    # on boucle jour par jour CAR LIMIT2 A DES TRANCHE DE 12H/24H MAXIMUM, 
    # on doit faire plusieurs appels pour couvrir toute la période
    while current_date <= end_date.date():
        date_str = current_date.strftime("%Y-%m-%d")
        is_today = (current_date == end_date.date())

        print(f"  Jour {date_str}  ")

        # todo : plus besoin de gerer le dernier jour,
        # on prendra juste la tranche 00:00-23:59 pour tous les jours, y compris le dernier qui est hier à 23:59
        if is_today:
            periods = [
                (f"{date_str}T00:00", f"{date_str}T11:59"),
                (f"{date_str}T12:00", end_date.strftime("%Y-%m-%dT%H:%M"))
            ]
        else:
            periods = [
                (f"{date_str}T00:00", f"{date_str}T11:59"),
                (f"{date_str}T12:00", f"{date_str}T23:59")
            ]
        # on fait l'appel api pour chaque tranche horaire de demi journée comme imposé par l'API
        for from_local, to_local in periods:
            for icao in icaos:
                url = f"{API_HOST}/airports/icao/{icao}/delays/{from_local}/{to_local}"

                try:
                    response = requests.get(url, headers=HEADERS, timeout=20)
                    
                    if response.status_code != 200:
                        logging.error(f"    {response.status_code} pour {icao} ({from_local} → {to_local})")
                        logging.error(f"    Message API : {response.text}")
                        continue

                    data = response.json()

                    if isinstance(data, dict):
                        data = [data]

                    # les données de l'API sont déjà très bien structurées, on peut faire un simple parsing pour les mettre dans un DataFrame
                    rows = []
                    for item in data:
                        row = {
                            "icao": icao,
                            "window_from_utc":   item.get("from", {}).get("utc"),
                            "window_to_utc":     item.get("to", {}).get("utc"),
                            "window_from_local": item.get("from", {}).get("local"),
                            "window_to_local":   item.get("to", {}).get("local"),

                            "dep_num_total":     item.get("departuresDelayInformation", {}).get("numTotal"),
                            "dep_num_qualified": item.get("departuresDelayInformation", {}).get("numQualifiedTotal"),
                            "dep_num_cancelled": item.get("departuresDelayInformation", {}).get("numCancelled"),
                            "dep_median_delay":  item.get("departuresDelayInformation", {}).get("medianDelay"),   # brut INTERVAL
                            "dep_delay_index":   item.get("departuresDelayInformation", {}).get("delayIndex"),

                            "arr_num_total":     item.get("arrivalsDelayInformation", {}).get("numTotal"),
                            "arr_num_qualified": item.get("arrivalsDelayInformation", {}).get("numQualifiedTotal"),
                            "arr_num_cancelled": item.get("arrivalsDelayInformation", {}).get("numCancelled"),
                            "arr_median_delay":  item.get("arrivalsDelayInformation", {}).get("medianDelay"),     # brut INTERVAL
                            "arr_delay_index":   item.get("arrivalsDelayInformation", {}).get("delayIndex"),
                        }
                        rows.append(row)

                    df = pd.DataFrame(rows)

                    if not df.empty:

                        # ON INSERE DIRECTEMENT EN BASE
                        inserer_base(engine, df, "airport_delays", conflict_columns=['icao', 'window_from_utc'])
                        total_inserted += len(df)
                        logging.info(f"    ✓ {len(df)} enregistrements bruts insérés pour {icao} ({from_local} → {to_local})")
                    else:
                        logging.info(f"    Pas de données pour {icao} ({from_local} → {to_local})")

                except Exception as e:
                    logging.error(f"    Erreur pour {icao} ({from_local} → {to_local}) : {e}")

                time.sleep(2.2)

        current_date += timedelta(days=1)

    logging.info(f"✅ Récupération terminée - Total : {total_inserted} enregistrements bruts insérés")
    return None

### API METEO (OPEN-WEATHER)

#### Script extraction des données depuis la request API OPEN-WEATHER

In [ ]:
def get_meteo_pour_tous_aeroports(
    coord_aeroport: dict,
    nb_jours: int,
    start_date: datetime,
    end_date: datetime,
    engine
):
    """
    Récupère la météo horaire pour tous les aéroports 
    sur la période définie par start_date + nb_jours.
    """

    unique_key = ['time', 'icao']  # clé pour éviter les doublons en base
    # Calcul de la période
    start_date_str = start_date.strftime("%Y-%m-%d")
    # end_date = start_date + timedelta(days=nb_jours - 1)
    end_date_str = end_date.strftime("%Y-%m-%d")

    logging.info(f"Récupération météo du {start_date_str} au {end_date_str} "
          f"({nb_jours} jours) pour {len(coord_aeroport)} aéroports\n")

    all_meteo = []

    for airport_code in coord_aeroport.keys():
        logging.info(f"=> Récupération météo pour {airport_code}")
        
        lat, lon = coord_aeroport[airport_code]
        
        url = "https://archive-api.open-meteo.com/v1/archive"
        
        params = {
            "latitude": lat,
            "longitude": lon,
            "start_date": start_date_str,
            "end_date": end_date_str,
            "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,pressure_msl,precipitation,cloud_cover,wind_gusts_10m",
            "timezone": "Europe/Paris"
        }
        
        try:
            r = requests.get(url, params=params, timeout=20)
            r.raise_for_status()
            data = r.json()
            
            if 'hourly' in data:
                df = pd.DataFrame(data['hourly'])
                df['time'] = pd.to_datetime(df['time'], utc=True)
                df['icao'] = airport_code
                inserer_base(engine, df, "meteo_aeroports", conflict_columns=unique_key)
                all_meteo.append(df)
                logging.info(f"   → {len(df)} heures récupérées pour {airport_code}")
                
        except Exception as e:
            logging.error(f"   Erreur API pour {airport_code}: {e}")
        
        time.sleep(1.0)  # respect du rate limit

    # Construction du dataframe final

    logging.info("=== Récupération météo pour tous les aéroports terminée ===\n")

    
    logging.info(f"Recuperation météo terminée")
    
    return None

***
## Récupération des données 

### Mouvements Aéroports 
#### -  Mouvements et statuts des vols des 5 principaux aéroports Français
#### -  Requête API AeroDataBox => Dataframe "df_mouvs"
(COMMANDE)

In [ ]:
logging.info("Début de la récupération des mouvements aéroports...")
logging.info(f"du {start_date} au {end_date}")

 #DEJA FAIT POUR CREER LES TABLES ET LES REMPLIRS,  BESOIN DE LELANCER UNE SEULE FOIS POUR TOUT CREER 
 
# init_table_mouvements_aeroports(engine)
# get_mouvements_aeroports_historical(
#     AEROPORTS, 
#     start_date,
#     end_date,
#     HEADERS, 
#     params,
#     engine=engine
# )
logging.info("fin de la récupération des mouvements aéroports.\n")


***
### Météo Aéroports 
#### -  Météo horaire sur toute la période pour chaque aéroport
#### -  Requête API Open-Weather => Dataframe "df_meteo"
(COMMANDE)

In [ ]:
logging.info("Début de la récupération des données météo pour les aéroports...")
logging.info(f"du {start_meteo_date} au {end_date}")

 #DEJA FAIT POUR CREER LES TABLES ET LES REMPLIRS,  BESOIN DE LELANCER UNE SEULE FOIS POUR TOUT CREER 
 
# init_table_meteo_aeroports(engine)
# get_meteo_pour_tous_aeroports(
#     coord_aeroport=COORDS_AEROPORTS,
#     nb_jours=nb_jours,
#     start_date=start_airports_delays,
#     end_date=end_date,
#     engine=engine
# )
logging.info("insertions des données météo terminées.\n")


***
### Delays Aéroports 
#### -  Retards par tranche de 2h apres les vols (mais beaucoup de Nan car uniquement lorsque beaucoup de traffic)
#### -  Requête API AeroDataBox => Dataframe "df_delays"
(COMMANDE)

In [ ]:
logging.info("Début de la récupération des données airport delays pour les aéroports...")
logging.info(f"du {start_date} au {end_date}")

 #DEJA FAIT POUR CREER LES TABLES ET LES REMPLIRS,  BESOIN DE LELANCER UNE SEULE FOIS POUR TOUT CREER 

# init_table_airport_delays(engine)
# get_airport_delays_historical(
#     coord_aeroport=COORDS_AEROPORTS,
#     start_date=start_airports_delays,
#     end_date=end_date,
#     engine=engine
# )

logging.info("insertions des données airport delays terminées.\n")

***
## Stockage sur AWS S3
(COMMANDE)

In [ ]:
def load_s3(df_mouvs, df_meteo, df_delays):
    timestamp = get_timestamp()
    logging.info(f"Timestamp pour les fichiers : {timestamp}")
    logging.info("Début de la sauvegarde des fichiers sur S3...")
    sauvegarder_s3(df_mouvs, f"df_mouvs_{nb_jours}jours_{timestamp}.parquet")
    sauvegarder_s3(df_meteo, f"df_meteo_{nb_jours}jours_{timestamp}.parquet")
    sauvegarder_s3(df_delays, f"df_airport_delays_{nb_jours}jours_{timestamp}.parquet")
    logging.info("Sauvegarde des fichiers sur S3 terminée.")

# Desactivé car déja stocké sur S3 et en Bdd
# on va lires des fichiers csv dans un df
# df_mouvs = pd.read_csv("data_csv/mouvements_aeroports.csv")
# df_meteo = pd.read_csv("data_csv/meteo_aeroports.csv")
# df_delays = pd.read_csv("data_csv/airport_delays.csv")
# load_s3(df_mouvs, df_meteo, df_delays)

***
## Insertions Base de Donnée NeonDB
Les insertions se font au fur et a mesure avec la meme fonction qui call Sont API 
=> moins de perte de données et moins de mouvements, moins de fichiers

***

***

<!-- ### recup du fichier pour le modele -->

## SUITE => Autres notebooks Etapes à suivre : Silver + Gold